### Using the Natural Language toolkit NLTK model for final data preparation and sentiment analysis

In [1]:
# importing libraries
import time
import pandas as pd
import nltk as nltk
from nltk.corpus import stopwords, words
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/stephenkeyen/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [2]:
import sys

if '/Users/stephenkeyen/Downloads/Functions 2' not in sys.path:
  sys.path.append('/Users/stephenkeyen/Downloads/Functions 2')

In [3]:
import function as fun

In [4]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

from sklearn.preprocessing import LabelEncoder
pd.set_option('display.width', 1000)

In [5]:
#loading Sentiment Intensity Analyzer
vader_sentiment = SentimentIntensityAnalyzer()

In [6]:
# There are 3 possibilities of input for a review:
# It could be "No Negative", in which case, return 0
# It could be "No Positive", in which case, return 0
# It could be a review, in which case calculate the sentiment
def calc_sentiment(review):    
    if review == "No Negative" or review == "No Positive":
        return 0
    return vader_sentiment.polarity_scores(review)["compound"] 

In [7]:
# Load the hotel reviews from CSV
df = pd.read_csv("/Users/stephenkeyen/NLP_projects/Hotel_Reviews/Hotel_Reviews_Final.csv")

In [8]:
# https://www.kaggle.com/ryanxjhan/fast-stop-words-removal # using the approach that Ryan recommends
start = time.time()
cache = set(stopwords.words("english"))
cache -= {'not', 'no', 'nor', 'never'} # these words are important for sentiment analysis, so we keep them
def remove_stopwords(review):
    text = " ".join([word for word in review.split() if word not in cache])
    return text

In [9]:
# Remove the stop words from both columns
df.Negative_Review = df.Negative_Review.apply(remove_stopwords)   
df.Positive_Review = df.Positive_Review.apply(remove_stopwords)

In [10]:

# Add a negative sentiment and positive sentiment column
print("Calculating sentiment columns for both positive and negative reviews")
start = time.time()
df["Negative_Sentiment"] = df.Negative_Review.apply(calc_sentiment)
df["Positive_Sentiment"] = df.Positive_Review.apply(calc_sentiment)
end = time.time()
print("Calculating sentiment took " + str(round(end - start, 2)) + " seconds")

Calculating sentiment columns for both positive and negative reviews
Calculating sentiment took 55.33 seconds


NLTK calc_sentiment function is implemented on both positive and negative review columns 


In [11]:
df = df.sort_values(by="Negative_Sentiment", ascending=True)
print(df[["Negative_Review", "Negative_Sentiment"]].head(10))

df = df.sort_values(by="Positive_Sentiment", ascending=True)
print(df[["Positive_Review", "Positive_Sentiment"]].head(10))

                                          Negative_Review  Negative_Sentiment
186202  So bad experience memories I hotel The first n...             -0.9933
306758  The staff Had bad experience even booking Janu...             -0.9927
159295  beetles black ladybirds spiders crawling walls...             -0.9916
450837  Scandalous hotel not 4 stars I never stay ever...             -0.9916
137801  No windows claustrophobic place even superior ...             -0.9909
382233  I strictly advised booking I needed twin room ...             -0.9909
129411  First charged twice room booked booking second...             -0.9907
127615  Our original room disgusting Underground suite...             -0.9900
201571  Everything DO NOT STAY AT THIS HOTEL I never i...             -0.9900
65536   This definitely not four star hotel Dirty toil...             -0.9898
                                          Positive_Review  Positive_Sentiment
5839    I completely disappointed mad since reception ...       

In [12]:
# Reorder the columns (This is cosmetic, but to make it easier to explore the data later)
df = df.reindex(["Hotel_Name", "Hotel_ID", "Hotel_Address", "Total_Reviews_Found", "Average_Score", "Reviewer_Score", "Negative_Sentiment", "Positive_Sentiment", "Reviewer_Nationality", "Leisure_trip", "Couple", "Solo_traveler", "Business_trip", "Group", "Family_with_young_children", "Family_with_older_children", "With_a_pet", "Negative_Review", "Positive_Review"], axis=1)

In [13]:
# # Saving new data file with calculated columns
# print("Saving results to Hotel_Reviews_NLP.csv")
# df.to_csv(r'/Users/stephenkeyen/NLP_projects/Hotel_Reviews/Hotel_Reviews_NLP.csv', index = False)

## Current dataset column statistics distribution
### Negative and positive sentiment scoring distribution is balanced but 50 to 70 percentile of negative comments are graded as neutral. further investigation can be done. Standard deviation is low statistical ranges can be trusted.

In [14]:
print(fun.metadata(df))

                   column_name datatype  missing_percent  unique        mean         std     min       25%       50%        75%        max
0                   Hotel_Name      str              0.0    1475         NaN         NaN     NaN       NaN       NaN        NaN        NaN
1                     Hotel_ID      str              0.0    1477         NaN         NaN     NaN       NaN       NaN        NaN        NaN
2                Hotel_Address      str              0.0       6         NaN         NaN     NaN       NaN       NaN        NaN        NaN
3          Total_Reviews_Found    int64              0.0     661  912.135555  892.139104  8.0000  312.0000  651.0000  1148.0000  4789.0000
4                Average_Score  float64              0.0      38    8.395813    0.610962  5.1000    8.1000    8.4000     8.8000     9.7000
5               Reviewer_Score  float64              0.0      37    8.395594    1.638170  2.5000    7.5000    8.8000     9.6000    10.0000
6           Negative_Sentim

## Top countries with the most negative reviews, using sentiment scoring average

In [15]:
nationality_negative = (
    df.groupby('Reviewer_Nationality')
      .agg(
          Total_Reviews=('Reviewer_Nationality', 'size'),
          Avg_Negative_Sentiment=('Negative_Sentiment', 'mean')
      )
      .sort_values('Avg_Negative_Sentiment', ascending=False)
)

print(nationality_negative.head(10))

                          Total_Reviews  Avg_Negative_Sentiment
Reviewer_Nationality                                           
Saint Barts                           3                0.451567
Guinea                                1                0.296000
Burundi                               3                0.290500
Cook Islands                          2                0.243400
Djibouti                              2                0.220200
Central Africa Republic               3                0.199633
Wallis and Futuna                     2                0.179300
Liberia                               3                0.152333
Togo                                  7                0.114457
Guam                                 14                0.095714


In [16]:
print(nationality_negative.head(20))

                                   Total_Reviews  Avg_Negative_Sentiment
Reviewer_Nationality                                                    
Saint Barts                                    3                0.451567
Guinea                                         1                0.296000
Burundi                                        3                0.290500
Cook Islands                                   2                0.243400
Djibouti                                       2                0.220200
Central Africa Republic                        3                0.199633
Wallis and Futuna                              2                0.179300
Liberia                                        3                0.152333
Togo                                           7                0.114457
Guam                                          14                0.095714
Saint Kitts and Nevis                          9                0.087867
Northern Mariana Islands                       1   

## Top countries with the most positive reviews, using sentiment scoring average

In [17]:
nationality_positive = (
    df.groupby('Reviewer_Nationality')
      .agg(
          Total_Reviews=('Reviewer_Nationality', 'size'),
          Avg_Positive_Sentiment=('Positive_Sentiment', 'mean')
      )
      .sort_values('Avg_Positive_Sentiment', ascending=False)
)
print(nationality_positive.head(20))

                            Total_Reviews  Avg_Positive_Sentiment
Reviewer_Nationality                                             
Cape Verde                              1                0.961100
Crimea                                  6                0.941300
Lesotho                                 3                0.886867
Tuvalu                                  1                0.883400
Saint Vincent Grenadines                2                0.882650
Cocos K I                               2                0.837000
Grenada                                 3                0.789633
Vatican City                            1                0.784500
Falkland Islands Malvinas               6                0.782383
Central Africa Republic                 3                0.765667
Montserrat                              2                0.741400
Guam                                   14                0.737171
South Sudan                             2                0.732600
Malawi    

In [19]:
hotel_negative = (
    df.groupby(['Hotel_ID', 'Hotel_Name', 'Hotel_Address'])
      .agg(
          Total_Reviews=('Hotel_ID', 'size'),
          Avg_Negative_Sentiment=('Negative_Sentiment', 'mean')
      )
      .sort_values('Avg_Negative_Sentiment', ascending=False)
)


## Hotels with the most negative reviews, including score, IDs, addresses and location

- Paris dominates because it is among the cities with the most hotel reviews collected

In [20]:
print(hotel_negative.head(20))

                                                                                                                      Total_Reviews  Avg_Negative_Sentiment
Hotel_ID                                           Hotel_Name                                 Hotel_Address                                                
XO Hotel_48.883_2.298                              XO Hotel                                   Paris, France                      13                0.169608
Drawing Hotel_48.864_2.336                         Drawing Hotel                              Paris, France                      14                0.149500
Bulgari Hotel Milano_45.47_9.19                    Bulgari Hotel Milano                       Milan, Italy                       17                0.135871
Maison Albar Hotel Paris C line_48.861_2.344       Maison Albar Hotel Paris C line            Paris, France                      62                0.121834
Am Spiegeln_48.154_16.283                          Am Spiegeln  

## Hotels with the most positive reviews, including score, IDs, addresses and location

- Pari also has dominates for the same reasons as the negative

In [21]:
hotel_positive = (
    df.groupby(['Hotel_ID', 'Hotel_Name', 'Hotel_Address'])
      .agg(
          Total_Reviews=('Hotel_ID', 'size'),
          Avg_Positive_Sentiment=('Positive_Sentiment', 'mean')
      )
      .sort_values('Avg_Positive_Sentiment', ascending=False)
)

In [22]:
print(hotel_positive.head(20))

                                                                                                                Total_Reviews  Avg_Positive_Sentiment
Hotel_ID                                          Hotel_Name                            Hotel_Address                                                
Hotel Eiffel Blomet_48.842_2.302                  Hotel Eiffel Blomet                   Paris, France                      15                0.831087
Room Mate Gerard_41.392_2.177                     Room Mate Gerard                      Barcelona, Spain                   14                0.819879
Hotel Whistler_48.879_2.356                       Hotel Whistler                        Paris, France                      28                0.816543
Pillows Anna van den Vondel Amsterdam_52.361_4.87 Pillows Anna van den Vondel Amsterdam Amsterdam, Netherlands             56                0.801279
Catalonia Magdalenes_41.386_2.175                 Catalonia Magdalenes                  Barcelona, S

In [27]:
tag_columns = [
    'Couple',
    'Solo_traveler',
    'Group',
    'Business_trip', 
    'Leisure_trip',
    'Family_with_young_children',
    'Family_with_older_children',
    'With_a_pet'
]

## Using Customer frequent tags to understand customer types based on tags and their average sentiments based on positive and nagative review scores

In [28]:
results = []

for tag in tag_columns:
    
    tag_df = df[df[tag] == 1]

    results.append({
        'Traveller_Type': tag,
        'Reviews': len(tag_df),
        'Avg_Positive_Sentiment': tag_df['Positive_Sentiment'].mean(),
        'Avg_Negative_Sentiment': tag_df['Negative_Sentiment'].mean()
    })

traveller_summary = (
    pd.DataFrame(results)
      .sort_values('Avg_Positive_Sentiment', ascending=False)
)

print(traveller_summary)

               Traveller_Type  Reviews  Avg_Positive_Sentiment  Avg_Negative_Sentiment
0                      Couple   250756                0.588612               -0.028902
4                Leisure_trip   415130                0.584147               -0.029205
6  Family_with_older_children    26196                0.581921               -0.036546
2                       Group    67071                0.569621               -0.031171
5  Family_with_young_children    60603                0.545558               -0.041694
7                  With_a_pet     1385                0.545105               -0.040935
1               Solo_traveler   107844                0.522616               -0.047142
3               Business_trip    82427                0.492332               -0.061578


### The averages based on customer tags are very similar but Couple tags seems to have the highest positive and negative averages, which is Valid when dealing with two individuals with different types of needs to be addressed, Hotels can consider prioritising making changes that include both male and female couple demographics and the process waters down to other customer tag types.

In [29]:
print(traveller_summary.head(1))

  Traveller_Type  Reviews  Avg_Positive_Sentiment  Avg_Negative_Sentiment
0         Couple   250756                0.588612               -0.028902
